# Week 02 — Testing, debugging and Git collaboration

**Seismic Python & AI Engineering | Hands-on notebook**

Reliable research code requires repeatable tests, explicit sign conventions, useful failure messages and focused commits—not just plausible plots.

**Estimated independent study:** 4–5 hours in this notebook, plus 2–3 hours on repository exercises and review.  
**Data:** entirely synthetic; no external API, proprietary dataset, or GPU required.  
**Prerequisite:** complete the root README setup and open Jupyter from the repository root.

### Learning outcomes
- Differentiate unit, integration and regression tests.
- Write pytest parameterized tests and use breakpoints effectively.
- Verify time-shift sign conventions and lag bounds.
- Follow a feature-branch and pull-request workflow.

### How to study
Read the explanations, execute the worked cells in order, complete the **Your turn** tasks, and implement the week's function in `src/seismic_toolkit/challenges.py`. Run the targeted exercise test to assess progress. The initial baseline tests should pass even before you complete the challenges.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from seismic_toolkit import __version__
print(f"Seismic toolkit version: {__version__}")
np.set_printoptions(precision=4, suppress=True)

## 1. Testing is an executable scientific claim
A plot that looks right is not a test. An assertion such as *“a monitor delayed by four samples yields +4 samples”* specifies a numerical outcome and a sign convention. Testing is especially important for AI-assisted code because generated code can be syntactically valid and numerically wrong.

In [ ]:
from seismic_toolkit.attributes import estimate_time_shift

dt = 0.002
baseline = np.zeros(128)
monitor = np.zeros(128)
baseline[50] = 1.0
monitor[54] = 1.0
shift_s = estimate_time_shift(baseline, monitor, dt=dt, max_lag=0.020)
print(f"Estimated monitor delay: {shift_s / dt:.0f} samples ({shift_s * 1000:.1f} ms)")
assert np.isclose(shift_s, 0.008)

## 2. Sign convention and lag restrictions
We define the delay of `monitor` **relative to** `baseline`: positive means later arrival. This is easy to reverse inadvertently when swapping the order of `scipy.signal.correlate` arguments.

**Your turn A:** repeat the experiment with a monitor three samples **earlier** than baseline, and with a maximal allowed lag of two samples. Explain why a plausible maximum correlation outside the allowed window must not be returned.

In [ ]:
early = np.zeros_like(baseline)
early[47] = 1.0
print("Early monitor shift (ms):", 1000 * estimate_time_shift(baseline, early, dt, 0.020))
print("Identical traces shift (ms):", 1000 * estimate_time_shift(baseline, baseline, dt, 0.020))
assert np.isclose(estimate_time_shift(baseline, early, dt, 0.020), -0.006)

## 3. Test design: arrange, act, assert
A good test has a descriptive name and a single objective. Use parametrization for a family of equivalent cases; use exceptions for invalid contracts. Avoid testing *how* your function is implemented, when you can test *what* it must return.

Example in `pytest`:

```python
@pytest.mark.parametrize("delay", [-5, -2, 0, 3, 6])
def test_shift_sign(delay):
    baseline = np.zeros(100)
    monitor = np.zeros(100)
    baseline[40] = monitor[40 + delay] = 1
    assert estimate_time_shift(baseline, monitor, 0.001, 0.02) == pytest.approx(delay * 0.001)
```

**Your turn B:** add a test for mismatched trace lengths and nonfinite inputs.

In [ ]:
for bad_monitor in [np.zeros(70), np.full(128, np.nan)]:
    try:
        estimate_time_shift(baseline, bad_monitor, dt, 0.02)
    except ValueError as exc:
        print(f"Expected validation failure: {exc}")

## 4. Debugging without guessing
When a test fails, reproduce it with the **smallest deterministic input**. Inspect array shapes, timestamps, units, input normalization, and the call stack. In VS Code, click next to a line to set a breakpoint, then choose **Debug Python File** or **Debug Test**.

An efficient debugging loop: (1) formulate an expected result, (2) reproduce the failure, (3) isolate inputs, (4) inspect variables, (5) patch only the cause, (6) rerun all regression tests.

In [ ]:
from scipy.signal import correlate, correlation_lags
corr = correlate(monitor, baseline, mode="full")
lags = correlation_lags(len(monitor), len(baseline), mode="full")
peak_index = int(np.argmax(corr))
print("Peak lag from correlate(monitor, baseline):", lags[peak_index])
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.plot(lags, corr)
ax.set(xlabel="Lag (samples)", ylabel="Cross-correlation", title="Correlation lag sign")
ax.grid(alpha=0.25)
plt.show()

## 5. Git working agreement
Create a branch for every feature or bug fix:

```bash
git switch -c feature/week02-shift-tests
python -m pytest -q
git add src tests notebooks
git commit -m "test: cover signed monitor time shifts"
git push -u origin feature/week02-shift-tests
```

Open a pull request, describe your change and attach test output. Do not commit `.venv/`, data with restricted licensing, `.env` files or AI API tokens. For a local practice repository, you can review your own pull request as if you were a teammate.

## 6. Independent implementation — Week 2 challenge
Implement `estimate_shift_samples(baseline, monitor, max_lag_samples)` in `challenges.py`. Return an integer signed shift in **samples**, positive if monitor is later. Reject negative lag, unequal-length traces and zero-energy data. Verify both a +4 and -4 delay.

**Applied problem:** add random noise to baseline/monitor (seed your random generator). At what noise level does your integer-lag estimate become unreliable? Do not confuse an empirical result on one trace with a universal noise threshold.

In [ ]:
rng = np.random.default_rng(2026)
noisy_baseline = baseline + 0.08 * rng.standard_normal(baseline.size)
noisy_monitor = monitor + 0.08 * rng.standard_normal(monitor.size)
print("Noisy shift (samples):", estimate_time_shift(noisy_baseline, noisy_monitor, dt, 0.020) / dt)
assert np.isfinite(noisy_baseline).all()

---

## Weekly submission and self-review

1. Complete the independent exercise in `src/seismic_toolkit/challenges.py`.
2. Run: `python -m pytest -o addopts='' -m exercise -k week2 -q`.
3. Run baseline regression tests: `python -m pytest -q`.
4. Commit a focused change to a feature branch; document what changed and why.
5. Record one failure mode, one design decision, and one next-step question in your learning journal.

**Further work:** see [`../exercises/week02.md`](../exercises/week02.md) for a graded problem, extra practice and acceptance criteria. Compare with `solutions/reference_challenges.py` only **after** attempting your own solution.

**Important:** all datasets and algorithms are educational demonstrations, not validated reservoir-interpretation systems.